In [46]:
import re
import fitz 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import cv2
from sklearn.feature_extraction import _stop_words # Import stop words list for text preprocessing

In [47]:
# Function to extract text from PDF using fitz(PyMuPDF)
def extract_text_from_pdf(pdf_document):
    text = ''
    try:
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            text += page.get_text()
    except Exception as e:
        print(f"Unable to access PDF content: {e}")
    return text

# Function to extract features from text
def extract_features(text):
    # Extract invoice number, date, and amount using regular expressions
    invoice_number = re.search(r'\bInvoice Number:\s*(\w+)\b', text, re.IGNORECASE)
    date = re.search(r'\bDate:\s*([0-9/.-]+)\b', text, re.IGNORECASE)
    amount = re.search(r'\bAmount:\s*\$?([0-9,]+\.\d{2})\b', text, re.IGNORECASE)
    
    # Filter out stop words
    words = re.findall(r'\b\w+\b', text.lower())
    keywords = [word for word in words if word not in _stop_words.ENGLISH_STOP_WORDS]
    
    features = {
        'keywords': keywords,
        'invoice_number': invoice_number.group(1) if invoice_number else '',
        'date': date.group(1) if date else '',
        'amount': amount.group(1) if amount else ''
    }
    return features


In [48]:
# Function to calculate cosine similarity
def calculate_cosine_similarity(vector1, vector2):
    return cosine_similarity(vector1, vector2)[0][0]

# Function to calculate Jaccard similarity
def calculate_jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

In [49]:
# Function to compare images using OpenCV
def calculate_image_similarity(image1, image2):
    if image1 is None or image2 is None:
        print("Unable to load image for comparison.")
        return 0
    
    #ORB to detect and compute keypoints and descriptors
    orb = cv2.ORB_create()
    kp1, des1 = orb.detectAndCompute(image1, None)
    kp2, des2 = orb.detectAndCompute(image2, None)
    
    # Match descriptors using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    
    # Sort matches by distance and calculate similarity score
    matches = sorted(matches, key=lambda x: x.distance)
    similarity_score = len(matches) / max(len(kp1), len(kp2))
    return similarity_score

In [50]:
# Function to convert PDF to image
def convert_pdf_to_image(pdf_document):
    try:
        images = [pdf_document.load_page(i).get_pixmap() for i in range(len(pdf_document))]
        combined_image = None
        for image in images:
            img = cv2.imdecode(np.frombuffer(image.tobytes(), np.uint8), cv2.IMREAD_GRAYSCALE)
            if combined_image is None:
                combined_image = img
            else:
                combined_image = cv2.vconcat([combined_image, img])
        return combined_image
    except Exception as e:
        print(f"Error converting PDF to image: {e}")
        return None

In [51]:
# Function to load invoices into an in-memory database
def load_database(database_docs):
    database = []
    for doc_path in database_docs:
        doc = fitz.open(doc_path)  # Open the PDF document
        text = extract_text_from_pdf(doc)
        if text.strip():  # Ensure non-empty text
            features = extract_features(text)
            if features['keywords']:  # Ensure keywords are not empty
                database.append({'document': doc, 'features': features})
    return database

In [52]:
# Function to find the most similar invoice
def find_most_similar_invoice(input_doc_path, database, use_image_similarity=False):
    input_document = fitz.open(input_doc_path)  # Open the input PDF document
    input_text = extract_text_from_pdf(input_document)
    input_features = extract_features(input_text)
    
    # Convert features to TF-IDF vectors
    corpus = [doc['features']['keywords'] for doc in database]
    corpus.append(input_features['keywords'])
    vectorizer = TfidfVectorizer().fit_transform([' '.join(doc) for doc in corpus])
    vectors = vectorizer.toarray()
    
    input_vector = vectors[-1].reshape(1, -1)
    database_vectors = vectors[:-1]
    
    # Calculate similarity scores
    cosine_scores = [calculate_cosine_similarity(input_vector, vec.reshape(1, -1)) for vec in database_vectors]
    
    # Calculate Jaccard similarity scores
    input_set = set(input_features['keywords'])
    jaccard_scores = [calculate_jaccard_similarity(input_set, set(doc['features']['keywords'])) for doc in database]
    
    # Combine similarity scores
    combined_scores = [(cos + jac) / 2 for cos, jac in zip(cosine_scores, jaccard_scores)]
    
    if use_image_similarity:
        input_image = convert_pdf_to_image(input_document)
        if input_image is not None:
            image_scores = [calculate_image_similarity(input_image, convert_pdf_to_image(doc['document'])) for doc in database]
            combined_scores = [(comb + img) / 2 for comb, img in zip(combined_scores, image_scores)]
    
    # Find the most similar invoice
    most_similar_index = np.argmax(combined_scores)
    return database[most_similar_index]['document'], combined_scores[most_similar_index]


In [53]:
# Main function to run the program
def main():
    database_docs = [
        'train/2024.03.15_0954.pdf', 
        'train/2024.03.15_1145.pdf',
        "train/invoice_102856.pdf", 
        "train/invoice_77073.pdf",
        "train/Faller_8.PDF"
        ]  # List of paths to existing invoices
    test_docs = [
        'test/invoice_77098.pdf',
        'test/invoice_102857.pdf',
    ] # Path to the input invoice
    
    database = load_database(database_docs)
    
    for input_doc_path in test_docs:
        most_similar_invoice, similarity_score = find_most_similar_invoice(input_doc_path, database, use_image_similarity=True)
        input_file_name = input_doc_path.split('/')[-1]  # Get the file name without the path
        similar_file_name = most_similar_invoice.name.split('/')[-1]  # Get the similar file name without the path
        print(f'File: {input_file_name}, Most similar invoice: {similar_file_name}, Similarity score: {similarity_score}')

if __name__ == '__main__':
    main()


File: invoice_77098.pdf, Most similar invoice: invoice_77073.pdf, Similarity score: 0.8974992053621179
File: invoice_102857.pdf, Most similar invoice: invoice_102856.pdf, Similarity score: 0.7809600457108951
